# Ingest sprints.json file
1. Read the all the files from the sprints folder using spark dataframe reader API
1. Define and enforce schema 
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table    

> Note: JSON is in multi line format

In [0]:
%run ../00-common-Config/01-environment-variable

In [0]:
%run ../00-common-Config/02-helper-function

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
source_file=f"{landing_path}/{v_batch_id}/sprints"
table_name=f"{catalog_name}.{bronze_schema}.sprints"

In [0]:
spark.read.format("json").option('multiline', 'true').load(source_file)

In [0]:

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType

sprints_schema = StructType([
    StructField("constructorId", StringType(), True),
    StructField("date", DateType(), True),  
    StructField("driverId", StringType(), True),
    StructField("grid", LongType(), True),
    StructField("laps", LongType(), True),
    StructField("number", LongType(), True),
    StructField("points", DoubleType(), True),
    StructField("position", LongType(), True),
    StructField("positionText", StringType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", LongType(), True),
    StructField("season", LongType(), True),
    StructField("status", StringType(), True),
    StructField("url", StringType(), True)
])


In [0]:
sprints_df = spark.read.format("json").schema(sprints_schema).option('multiline', 'true').load(source_file)

sprints_df.show()

In [0]:
sprints_df=add_ingestion_metadata(sprints_df)

In [0]:
# sprints_df.write.format("delta")\
#     .mode("overwrite")\
#     .saveAsTable(table_name)

write_to_bronze(
    input_df=sprints_df,
    target_table=table_name,
    batch_id=v_batch_id
)


In [0]:
spark.read.table(table_name).display()